In [1]:
# Import libraries
import pandas as pd

# Load processed data
agg_df = pd.read_csv("../data/processed/agg_data.csv")

In [2]:
# Convert Month back to datetime
agg_df['Month'] = pd.to_datetime(agg_df['Month'])
agg_df = agg_df.sort_values(['StockCode', 'Month'])

In [3]:
# Create lag features
# Previous month demand
agg_df['lag_1_units'] = agg_df.groupby('StockCode')['UnitsSold'].shift(1)

# Rolling average demand
agg_df['rolling_3_units'] = agg_df.groupby('StockCode')['UnitsSold'].rolling(3).mean().reset_index(0, drop=True)

In [4]:
# Price change feature
agg_df['price_change_pct'] = agg_df.groupby('StockCode')['AvgPrice'].pct_change()

# Time features
agg_df['month_num'] = agg_df['Month'].dt.month
agg_df['year'] = agg_df['Month'].dt.year

In [5]:
# Drop NaNs
model_df = agg_df.dropna().copy()

In [6]:
# Filter to top products
top_products = (
    model_df.groupby('StockCode')['UnitsSold']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index
)

model_df = model_df[model_df['StockCode'].isin(top_products)]

model_df.head()

,StockCode,Month,UnitsSold,AvgPrice,Revenue,lag_1_units,rolling_3_units,price_change_pct,month_num,year
92,15036,2011-02-01,2172,0.740476,1449.00,912.0,1072.000000,0.012040,2,2011
93,15036,2011-03-01,2912,0.735000,1986.00,2172.0,1998.666667,-0.007395,3,2011
94,15036,2011-04-01,1900,0.813667,1431.68,2912.0,2328.000000,0.107029,4,2011
95,15036,2011-05-01,3888,0.820000,2921.04,1900.0,2900.000000,0.007784,5,2011
96,15036,2011-06-01,3096,0.819800,2323.08,3888.0,2961.333333,-0.000244,6,2011


In [7]:
# Save model data
model_df.to_csv("../data/processed/model_data.csv", index=False)